# Experiment 4: coupled double-well / two-component phi4 field

This notebook defines a periodic two-component field with coupled local
double-well potential

```text
W(x,y) = ax/4*(x^2-1)^2 + ay/4*(y^2-1)^2
         + c*x*y + hx*x + hy*y + eta/2*x^2*y.
```

The local potential has four sign-state minima.  A gradient-flow basin map in
the local two-dimensional plane defines site-wise phase labels, while the mean
order parameter is mapped through the same basin lookup to define a global
field phase.  The notebook compares Langevin with Levy-score corrected shell
jumps on MST, cycle, five-edge, and complete four-phase graphs.


## Scientific workflow and outputs

The notebook verifies the four local minima and Hessians, constructs the local
basin map and local basin masses, derives a coherent-field Laplace phase-mass
reference, checks the moment-optimized score against direct energy
differences, runs the graph-family comparison, and reports phase-reference TV,
communication, physical field observables, and stabilization diagnostics.


In [ ]:
import os, math, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.optimize import minimize

PROFILE = os.environ.get("LEVY_PROFILE", "paperlite").lower()
GLOBAL_SEED = 20260123

def ensure_dir(p):
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p

def find_project_root(start=None):
    path = Path.cwd() if start is None else Path(start).resolve()
    while path.name != "levy-score-sampling-project" and path.parent != path:
        path = path.parent
    if path.name != "levy-score-sampling-project":
        raise RuntimeError("Could not locate levy-score-sampling-project from current working directory")
    return path

PROJECT_ROOT = find_project_root()
RELEASE_ROOT = ensure_dir(PROJECT_ROOT / "manuscript_clean_active" / "numerics" / "four_experiment_release")
RELEASE_TABLE_DIR = ensure_dir(RELEASE_ROOT / "tables")
RELEASE_LOG_DIR = ensure_dir(RELEASE_ROOT / "logs")
RELEASE_FIG_ROOT = ensure_dir(PROJECT_ROOT / "manuscript_clean_active" / "figures" / "four_experiment_release")
MAIN_FIG_DIR = ensure_dir(RELEASE_FIG_ROOT / "main_candidates")
APPENDIX_FIG_DIR = ensure_dir(RELEASE_FIG_ROOT / "appendix_candidates")
DIAGNOSTIC_FIG_DIR = ensure_dir(RELEASE_FIG_ROOT / "diagnostics")
MANUSCRIPT_FIG_DIR = DIAGNOSTIC_FIG_DIR
CANDIDATE_RESULT_DIR = RELEASE_ROOT
OUTDIR = ensure_dir(CANDIDATE_RESULT_DIR / "vector_gl_coupled_phi4")
FIGDIR = MANUSCRIPT_FIG_DIR
FIG_PREFIX = "vector_gl_"
TABDIR = ensure_dir(RELEASE_TABLE_DIR / "04_coupled_phi4_gl")
LOGDIR = RELEASE_LOG_DIR

# Euclidean shell half-widths for shell-jump sensitivity checks.
SHELL_WIDTH_CANDIDATES = [0.0, 0.08, 0.2, 0.4, 0.8]

PROFILE_ALIASES = {"smoke": "debug"}
PROFILE = PROFILE_ALIASES.get(PROFILE, PROFILE)

def savefig(fig, name, dpi=220):
    try:
        fig.tight_layout()
    except Exception:
        pass
    stem = str(name)
    if not stem.startswith(FIG_PREFIX):
        stem = FIG_PREFIX + stem
    pdf = FIGDIR / f"{stem}.pdf"
    fig.savefig(pdf, bbox_inches="tight")
    print("saved", pdf)
    return pdf


def method_color(m): return canonical_method_color(m)
def method_marker(m): return canonical_method_marker(m)

def clean_axes(ax, grid=True):
    if grid: ax.grid(alpha=0.25, linewidth=0.6)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

def panel_label(ax, label, x=-0.12, y=1.04):
    ax.text(x, y, label, transform=ax.transAxes, fontsize=13, fontweight="bold", va="bottom", ha="left")

PROFILE_CONFIGS = {
    "debug": dict(n_grid=6, n_particles=16, n_steps=12, dt=0.0030,
                  n_seeds=1, record_every=3, theta_n=3, rho_n=3, h_shell=0.2, jump_lam=1.6,
                  n_restarts=2, n_jobs=1, mixing_tol=0.20),
    "paperlite": dict(n_grid=32, n_particles=900, n_steps=1300, dt=0.0020,
                      n_seeds=3, record_every=20, theta_n=5, rho_n=3, h_shell=0.08, jump_lam=1.6,
                      n_restarts=0, n_jobs=1, mixing_tol=0.10),
    "paperlite_extended": dict(n_grid=32, n_particles=900, n_steps=1300, dt=0.0020,
                               n_seeds=3, record_every=20, theta_n=5, rho_n=3, h_shell=0.08, jump_lam=1.6,
                               n_restarts=0, n_jobs=1, mixing_tol=0.10),
    "paper": dict(n_grid=48, n_particles=3000, n_steps=3200, dt=0.0015,
                  n_seeds=5, record_every=32, theta_n=7, rho_n=7, h_shell=0.08, jump_lam=1.6,
                  n_restarts=0, n_jobs=1, mixing_tol=0.08),
}
if PROFILE not in PROFILE_CONFIGS:
    raise ValueError(f"Unknown LEVY_PROFILE={PROFILE!r}; expected one of {sorted(PROFILE_CONFIGS)}")
CFG = PROFILE_CONFIGS[PROFILE]

n_modes = 4
n_grid = CFG["n_grid"]
dim = 2*n_grid
h = 1.0/n_grid
kappa = 2.5
eps = 0.10
box_lo, box_hi = -1.75, 1.75
grid_x = np.arange(n_grid)/n_grid

PROFILE_METHODS = {
    "debug": ["Langevin", "LSC-MST-shell", "LSC-cycle-shell", "LSC-5-shell", "LSC-complete-shell"],
    "paperlite": ["Langevin", "LSC-MST-shell", "LSC-cycle-shell", "LSC-5-shell", "LSC-complete-shell"],
    "paperlite_extended": ["Langevin", "LSC-MST-shell", "LSC-cycle-shell", "LSC-5-shell", "LSC-complete-shell"],
    "paper": ["Langevin", "LSC-MST-shell", "LSC-cycle-shell", "LSC-5-shell", "LSC-complete-shell"],
}

print("PROFILE =", PROFILE)
print("n_grid =", n_grid, "state dimension =", dim, "n_modes =", n_modes)
print("periodic lattice: u_{n+1}=u_1")
print("kappa =", kappa, "epsilon =", eps, "total jump rate =", CFG["jump_lam"])
print("quadrature theta_n =", CFG["theta_n"], "rho_n =", CFG["rho_n"])
print("checkpoint frequency record_every =", CFG["record_every"])
print("per-method timeout seconds =", os.environ.get("LEVY_GL_METHOD_TIMEOUT_SECONDS", "1800"))
print("methods =", PROFILE_METHODS[PROFILE])


In [ ]:
# Phase 16A shared theory-driven utilities.
import sys

RELEASE_COMMON_ROOT = RELEASE_ROOT / "common"
if str(RELEASE_COMMON_ROOT) not in sys.path:
    sys.path.insert(0, str(RELEASE_COMMON_ROOT))

from levy.jumps import (
    AtomJump,
    EdgeShellJump,
    apply_compound_poisson_shell_jumps,
)
from levy.metrics import (
    cost_proxy_fields,
    compute_jump_step_transition_matrix,
    compute_recorded_phase_transition_matrix,
    mode_entropy_metrics,
    transition_rows_from_counts,
)
from levy.diagnostics import energy_before_after, jump_count_summary, merge_score_diagnostics
from levy.gl_sequential import (
    gl_method_output_paths,
    gl_progress_path,
    write_gl_progress,
)
from levy.phi4 import (
    DEFAULT_COUPLED_PHI4_PARAMS,
    SIGN_PHASES,
    build_coupled_phi4_basin_map,
    coupled_phi4_graph_edge_sets,
    coupled_phi4_local_gibbs_basin_masses,
    coupled_phi4_local_potential,
    coupled_phi4_homogeneous_shift_local_energy_delta,
    coupled_phi4_homogeneous_shift_local_energy_delta_moment,
    edge_direction_audit,
    edge_vectors_from_minima,
    find_coupled_phi4_minima,
    gl_energy_parts,
    levy_score_coupled_phi4_edge_shell,
    lookup_coupled_phi4_basin_labels,
    grad_coupled_phi4_local_potential,
    hess_coupled_phi4_local_potential,
    magnetization,
    minimum_parallel_sine,
    structure_factor,
    susceptibility,
    vector_correlation,
)

LOCAL_PARAMS = dict(DEFAULT_COUPLED_PHI4_PARAMS)

def legendre_unit_interval(n):
    x, w = np.polynomial.legendre.leggauss(int(n))
    return 0.5 * (x + 1.0), 0.5 * w

def shell_uniform_quadrature(h_shell, n_rho):
    x, w = np.polynomial.legendre.leggauss(int(n_rho))
    return float(h_shell) * x, 0.5 * w

print("local potential parameters:", LOCAL_PARAMS)


from levy.plot_style import (
    METHOD_STYLES as CANONICAL_METHOD_STYLES,
    apply_plot_style,
    method_color as canonical_method_color,
    method_marker as canonical_method_marker,
)
apply_plot_style(plt)
METHOD_COLORS = {name: style.color for name, style in CANONICAL_METHOD_STYLES.items()}
METHOD_MARKERS = {name: style.marker for name, style in CANONICAL_METHOD_STYLES.items()}


In [ ]:
# ---------------------------------------------------------------------
# Periodic coupled double-well vector phi4 energy, gradient, Hessian, and basin labels.
# ---------------------------------------------------------------------
def pack(q):
    q = np.asarray(q)
    return q.reshape(q.shape[:-2] + (2*n_grid,))

def unpack(z):
    z = np.asarray(z)
    return z.reshape(z.shape[:-1] + (n_grid, 2))

def local_W(q):
    return coupled_phi4_local_potential(q, **LOCAL_PARAMS)

def grad_local_W(q):
    return grad_coupled_phi4_local_potential(q, **LOCAL_PARAMS)

def hess_local_W(q):
    return hess_coupled_phi4_local_potential(q, **LOCAL_PARAMS)

def V_landau(z):
    z = np.asarray(z, dtype=float)
    q = unpack(z)
    diff = np.roll(q, -1, axis=-2) - q
    coupling = kappa/(2*h) * np.sum(diff*diff, axis=(-2,-1))
    local = h * np.sum(local_W(q), axis=-1)
    return coupling + local

def grad_V_landau(z):
    z = np.asarray(z, dtype=float)
    q = unpack(z)
    lap = 2*q - np.roll(q, 1, axis=-2) - np.roll(q, -1, axis=-2)
    g = (kappa/h)*lap + h*grad_local_W(q)
    return pack(g)

def hess_V_landau(z):
    z = np.asarray(z, dtype=float)
    q = unpack(z).reshape(n_grid, 2)
    H = np.zeros((dim, dim))
    I2 = np.eye(2)
    for i in range(n_grid):
        sl = slice(2*i, 2*i+2)
        H[sl, sl] += (2*kappa/h)*I2 + h*hess_local_W(q[i])
        ip = (i+1) % n_grid
        im = (i-1) % n_grid
        H[sl, slice(2*ip,2*ip+2)] += -(kappa/h)*I2
        H[sl, slice(2*im,2*im+2)] += -(kappa/h)*I2
    return H

def logp(z): return -V_landau(z)/eps
def grad_logp(z): return -grad_V_landau(z)/eps

phase_vectors, local_minima_rows = find_coupled_phi4_minima(LOCAL_PARAMS, minimize)
phase_names = list(SIGN_PHASES)
def phase_state(v): return np.tile(np.asarray(v, dtype=float), (n_grid,1)).reshape(-1)
phase_states = np.array([phase_state(v) for v in phase_vectors])

BASIN_MAP_CACHE = TABDIR / "coupled_phi4_basin_map_cache.npz"
if BASIN_MAP_CACHE.exists():
    cache = np.load(BASIN_MAP_CACHE, allow_pickle=True)
    basin_gx = cache["gx"]
    basin_gy = cache["gy"]
    basin_map_labels = cache["basin_labels"]
    local_2d_basin_mass = cache["basin_masses"].astype(float)
    basin_map_metadata = dict(zip(cache["metadata_keys"].astype(str), cache["metadata_values"].astype(str)))
else:
    basin_gx, basin_gy, basin_map_labels, basin_map_metadata = build_coupled_phi4_basin_map(
        LOCAL_PARAMS, phase_vectors, grid_n=300, step=0.08, max_iter=700, tol=1e-6
    )
    local_2d_basin_mass, _ = coupled_phi4_local_gibbs_basin_masses(
        basin_gx, basin_gy, basin_map_labels, LOCAL_PARAMS, eps, n_modes=n_modes
    )
    np.savez(
        BASIN_MAP_CACHE,
        gx=basin_gx,
        gy=basin_gy,
        basin_labels=basin_map_labels,
        minima_locations=phase_vectors,
        minima_energies=np.array([row["energy"] for row in local_minima_rows]),
        basin_masses=local_2d_basin_mass,
        param_names=np.array(list(LOCAL_PARAMS.keys())),
        param_values=np.array(list(LOCAL_PARAMS.values()), dtype=float),
        metadata_keys=np.array(list(basin_map_metadata.keys())),
        metadata_values=np.array([str(v) for v in basin_map_metadata.values()]),
    )

def mean_order_parameter(z): return np.mean(unpack(z), axis=-2)

def classify_phase(z):
    mean_q = mean_order_parameter(z)
    return lookup_coupled_phi4_basin_labels(mean_q, basin_gx, basin_gy, basin_map_labels)

def nearest_minimum_diagnostic_labels(z):
    z = np.asarray(z)
    D = np.linalg.norm(z[:,None,:] - phase_states[None,:,:], axis=2)
    return np.argmin(D, axis=1)

def site_basin_labels(z):
    q = unpack(z)
    flat = q.reshape(-1, 2)
    return lookup_coupled_phi4_basin_labels(flat, basin_gx, basin_gy, basin_map_labels).reshape(q.shape[:-1])

def basin_domain_wall_density(z):
    s = site_basin_labels(z)
    walls = s != np.roll(s, -1, axis=1)
    return walls.mean(axis=1)

def phase_masses(z):
    labs = classify_phase(z)
    return np.bincount(labs, minlength=n_modes)/len(labs)

def nearest_minimum_masses(z):
    labs = nearest_minimum_diagnostic_labels(z)
    return np.bincount(labs, minlength=n_modes)/len(labs)

def finite_difference_gradient_check(z, n_dirs=8, hfd=1e-5, seed=0):
    rng = np.random.default_rng(seed); z = np.asarray(z, dtype=float)
    g = grad_V_landau(z[None,:])[0]
    max_abs = max_rel = 0.0
    for _ in range(n_dirs):
        v = rng.normal(size=z.shape); v /= np.linalg.norm(v)
        fd = (V_landau((z+hfd*v)[None,:])[0]-V_landau((z-hfd*v)[None,:])[0])/(2*hfd)
        gd = float(np.dot(g,v))
        max_abs = max(max_abs, abs(fd-gd))
        max_rel = max(max_rel, abs(fd-gd)/(abs(fd)+abs(gd)+1e-12))
    return float(max_abs), float(max_rel)

# Coherent-field Laplace reference for the global phase masses.
coherent_phase_energies = V_landau(phase_states)
coherent_phase_logdet = []
for state in phase_states:
    H = 0.5*(hess_V_landau(state) + hess_V_landau(state).T)
    sign, logdet = np.linalg.slogdet(H)
    if sign <= 0:
        raise RuntimeError("Coherent-field Hessian is not positive definite")
    coherent_phase_logdet.append(float(logdet))
coherent_phase_logdet = np.asarray(coherent_phase_logdet)
phase_logweights = -coherent_phase_energies/eps - 0.5*coherent_phase_logdet
phase_weights = np.exp(phase_logweights - np.max(phase_logweights))
target_phase_mass = phase_weights / phase_weights.sum()

print("single-site Gibbs basin probabilities:", dict(zip(phase_names, local_2d_basin_mass)))
print("coherent-field Laplace phase-mass reference:", dict(zip(phase_names, target_phase_mass)))
print("basin map metadata:", basin_map_metadata)


In [ ]:
# ---------------------------------------------------------------------
# Lightweight local-potential, basin-map, and coherent-field minima check.
# ---------------------------------------------------------------------
local_minima_df = pd.DataFrame(local_minima_rows)
edge_rows = edge_vectors_from_minima(phase_vectors, phase_names)
edge_df = pd.DataFrame(edge_rows)
edge_audit = edge_direction_audit(edge_rows)
local_mass_rows = [dict(phase=phase_names[k], local_2d_basin_mass=float(local_2d_basin_mass[k])) for k in range(n_modes)]
local_check = pd.DataFrame([dict(
    n_minima=int(len(local_minima_df)),
    all_optimizer_success=bool(local_minima_df["optimizer_success"].all()),
    all_sign_states_ok=bool(local_minima_df["sign_state_ok"].all()),
    min_hessian_eig=float(local_minima_df["hessian_min_eig"].min()),
    min_parallel_sine=float(edge_audit["minimum_sine"]),
    closest_parallel_edge_a=edge_audit["closest_edge_direction_pair_a"],
    closest_parallel_edge_b=edge_audit["closest_edge_direction_pair_b"],
    min_local_2d_basin_mass=float(np.min(local_2d_basin_mass)),
    max_local_2d_basin_mass=float(np.max(local_2d_basin_mass)),
    no_well_destroyed=bool(len(local_minima_df) == 4 and local_minima_df["sign_state_ok"].all() and (local_minima_df["hessian_min_eig"] > 0.0).all()),
    local_2d_basin_masses_visible=bool(np.min(local_2d_basin_mass) >= 0.03 and np.max(local_2d_basin_mass) <= 0.80),
    edge_geometry_tradeoff=bool(edge_audit["minimum_sine"] < 0.15),
)]) 

local_minima_df.to_csv(TABDIR/"coupled_phi4_local_minima.csv", index=False)
edge_df.to_csv(TABDIR/"coupled_phi4_edge_vectors.csv", index=False)
pd.DataFrame([edge_audit]).to_csv(TABDIR/"coupled_phi4_edge_direction_audit.csv", index=False)
pd.DataFrame(local_mass_rows).to_csv(TABDIR/"coupled_phi4_local_2d_basin_masses.csv", index=False)
local_check.to_csv(TABDIR/"coupled_phi4_local_check.csv", index=False)
display(local_minima_df)
display(edge_df)
display(pd.DataFrame(local_mass_rows))
display(local_check)

assert len(local_minima_df) == 4
assert bool(local_check["all_optimizer_success"].iloc[0])
assert bool(local_check["all_sign_states_ok"].iloc[0])
assert bool(local_check["no_well_destroyed"].iloc[0])
assert bool(local_check["local_2d_basin_masses_visible"].iloc[0])

min_vals = V_landau(phase_states)
verify_rows=[]
for k,m in enumerate(phase_states):
    g=grad_V_landau(m[None,:])[0]
    H=0.5*(hess_V_landau(m)+hess_V_landau(m).T)
    eigs=np.linalg.eigvalsh(H)
    fdg_abs,fdg_rel=finite_difference_gradient_check(m, seed=GLOBAL_SEED+11*k)
    verify_rows.append(dict(mode=k, phase=phase_names[k], energy=float(min_vals[k]),
        local_x=float(phase_vectors[k,0]), local_y=float(phase_vectors[k,1]),
        local_2d_basin_mass_reference=float(local_2d_basin_mass[k]),
        grad_linf=float(np.max(np.abs(g))), grad_l2=float(np.linalg.norm(g)),
        hess_min_eig=float(np.min(eigs)), hess_max_eig=float(np.max(eigs)),
        fd_grad_max_abs_error=fdg_abs, fd_grad_max_rel_error=fdg_rel,
        stationarity_ok=bool(np.max(np.abs(g))<1e-8),
        hessian_positive_ok=bool(np.min(eigs)>1e-8), fd_gradient_ok=bool(fdg_abs<1e-6)))
minima_verification=pd.DataFrame(verify_rows)
minima_verification["quality_ok"]=minima_verification[["stationarity_ok","hessian_positive_ok","fd_gradient_ok"]].all(axis=1)
minima_verification.to_csv(TABDIR/"vector_gl_minima_verification.csv", index=False)
phase_reference_df = pd.DataFrame({
    "phase": phase_names,
    "coherent_field_energy": coherent_phase_energies,
    "coherent_field_hessian_logdet": coherent_phase_logdet,
    "laplace_phase_mass_reference": target_phase_mass,
})
phase_reference_df.to_csv(TABDIR/"coupled_phi4_global_phase_mass_reference.csv", index=False)
display(minima_verification)
display(phase_reference_df)


In [ ]:
# ---------------------------------------------------------------------
# Target geometry: local potential, gradient-flow basins, masses, and phases.
# ---------------------------------------------------------------------
q_mins = unpack(phase_states)
q1_grid = np.linspace(-1.55, 1.55, 260)
q2_grid = np.linspace(-1.55, 1.55, 260)
Q1, Q2 = np.meshgrid(q1_grid, q2_grid)
Wgrid = local_W(np.stack([Q1, Q2], axis=-1))

fig, ax = plt.subplots(2, 3, figsize=(16.2, 9.0), constrained_layout=True)
cs = ax[0,0].contourf(Q1, Q2, Wgrid, levels=45, cmap="viridis")
ax[0,0].scatter(phase_vectors[:,0], phase_vectors[:,1], c=np.arange(n_modes),
                cmap="tab10", edgecolor="black", s=70)
for k, p in enumerate(phase_vectors):
    ax[0,0].text(p[0], p[1]+0.08, phase_names[k], ha="center", fontsize=8)
ax[0,0].set_aspect("equal")
ax[0,0].set_title("coupled local potential W")
fig.colorbar(cs, ax=ax[0,0], fraction=0.046)

ax[0,1].contourf(basin_gx, basin_gy, basin_map_labels,
                 levels=np.arange(n_modes+1)-0.5, cmap="tab10", alpha=0.72)
ax[0,1].contour(Q1, Q2, Wgrid, levels=16, colors="k", linewidths=0.35, alpha=0.45)
ax[0,1].scatter(phase_vectors[:,0], phase_vectors[:,1], c="white", edgecolor="black", s=65)
for k, p in enumerate(phase_vectors):
    ax[0,1].text(p[0], p[1]+0.08, phase_names[k], ha="center", fontsize=8)
ax[0,1].set_aspect("equal")
ax[0,1].set_title("gradient-flow local basin partition")

x = np.arange(n_modes)
ax[0,2].bar(x, local_2d_basin_mass, color=plt.get_cmap("tab10")(x), alpha=0.82)
ax[0,2].set_xticks(x)
ax[0,2].set_xticklabels(phase_names)
ax[0,2].set_ylabel("probability mass")
ax[0,2].set_title("single-site Gibbs basin probabilities")

ax[1,0].bar(x, local_minima_df["hessian_min_eig"], color=plt.get_cmap("tab10")(x), alpha=0.82)
ax[1,0].axhline(0.0, color="k", lw=0.8)
ax[1,0].set_xticks(x)
ax[1,0].set_xticklabels(phase_names)
ax[1,0].set_ylabel("minimum eigenvalue")
ax[1,0].set_title("positive local Hessians")

for k in range(n_modes):
    ax[1,1].plot(grid_x, q_mins[k,:,0], lw=1.8, label=phase_names[k])
    ax[1,2].plot(grid_x, q_mins[k,:,1], lw=1.8, label=phase_names[k])
for axi, title, ylabel in [
    (ax[1,1], "coherent phase profiles: x component", "u1"),
    (ax[1,2], "coherent phase profiles: y component", "u2"),
]:
    axi.set_xlabel("lattice coordinate")
    axi.set_ylabel(ylabel)
    axi.set_title(title)
    axi.legend(frameon=False, fontsize=8, ncol=2)

for j, axi in enumerate(ax.ravel()):
    clean_axes(axi, grid=(j not in [0, 1]))
    panel_label(axi, "abcdef"[j])
savefig(fig, "fig00_coupled_phi4_geometry")
plt.show()

phase_profile_rows = []
for k in range(n_modes):
    for i, xcoord in enumerate(grid_x):
        phase_profile_rows.append(dict(
            phase=phase_names[k], site=i, lattice_x=float(xcoord),
            u1=float(q_mins[k,i,0]), u2=float(q_mins[k,i,1]),
        ))
pd.DataFrame(phase_profile_rows).to_csv(TABDIR/"vector_gl_coupled_phi4_phase_profiles.csv", index=False)


In [ ]:
# ---------------------------------------------------------------------
# Four-well jump graph laws and graph gaps.
# ---------------------------------------------------------------------
def atoms_from_undirected_edges(edges, name):
    atoms = []
    directed = []
    for i, j in edges:
        atoms.append(phase_states[j] - phase_states[i]); directed.append((i, j))
        atoms.append(phase_states[i] - phase_states[j]); directed.append((j, i))
    atoms = np.asarray(atoms)
    weights = np.ones(len(atoms)) / len(atoms)
    return AtomJump(atoms, weights, CFG["jump_lam"], name=name, edge_count=len(edges)), directed

def shell_from_undirected_edges(edges, name):
    atoms, directed = atoms_from_undirected_edges(edges, name + "-centers")
    return EdgeShellJump(atoms.atoms, atoms.weights, CFG["jump_lam"], CFG["h_shell"], name=name, edge_count=len(edges)), directed

def graph_gap_from_edges(edges, lam):
    m = len(edges); Q = np.zeros((n_modes,n_modes)); rate = lam/(2*m)
    for i,j in edges:
        Q[i,j] += rate; Q[j,i] += rate
    Q[np.diag_indices(n_modes)] = -Q.sum(axis=1)
    vals = np.linalg.eigvalsh(-Q)
    vals = np.sort(vals)
    return float(vals[1]), Q

graph_edge_sets = coupled_phi4_graph_edge_sets(phase_vectors)
print("four-phase graph edge sets:")
for gname, edges in graph_edge_sets.items():
    print(f"  {gname}: {[(phase_names[i], phase_names[j]) for i,j in edges]}")

jump_laws = {}
directed_edges_by_graph = {}
graph_rows = []
for gname, edges in graph_edge_sets.items():
    shell_law, directed = shell_from_undirected_edges(edges, f"{gname}-shell")
    directed_edges_by_graph[gname] = directed
    gap, Q = graph_gap_from_edges(edges, CFG["jump_lam"])
    method = f"LSC-{gname}-shell"
    jump_laws[method] = shell_law
    graph_rows.append(dict(method=method, graph_family=gname, jump_type=shell_law.jump_type,
                           undirected_edges=len(edges), directed_atoms=2*len(edges),
                           graph_gap=gap, total_jump_rate=CFG["jump_lam"],
                           h_shell=float(getattr(shell_law, "h_shell", 0.0)),
                           edge_set=";".join(f"{phase_names[i]}-{phase_names[j]}" for i,j in edges)))
graph_df = pd.DataFrame(graph_rows)
graph_df.to_csv(TABDIR/"vector_gl_graph_gaps.csv", index=False)
display(graph_df)

theta_nodes, theta_weights = legendre_unit_interval(CFG["theta_n"])
rho_nodes, rho_weights = shell_uniform_quadrature(CFG["h_shell"], CFG["rho_n"])

def score_for_jump_gl(z, jump):
    # Moment-based exact local-energy deltas; no gradient-energy recomputation.
    return levy_score_coupled_phi4_edge_shell(
        z, jump, n_grid=n_grid, local_params=LOCAL_PARAMS, eps=eps,
        theta_nodes=theta_nodes, theta_weights=theta_weights,
        rho_nodes=rho_nodes, rho_weights=rho_weights,
        log_clip=None, score_clip=1.0e8, return_diagnostics=True,
    )

def apply_jump_for_law_gl(z, jump, rng, dt):
    return apply_compound_poisson_shell_jumps(z, jump, rng, dt)

def safety_clip(z):
    z2 = np.clip(z, box_lo, box_hi)
    return z2, float(np.mean(z2 != z))

def tamed_euler_step(z, drift, rng, dt):
    drift_tamed = drift/(1.0+dt*np.linalg.norm(drift,axis=1,keepdims=True))
    return z + dt*drift_tamed + np.sqrt(2*eps*dt)*rng.standard_normal(z.shape)


In [ ]:
# ---------------------------------------------------------------------
# Sequential simulation: Langevin and four-phase graph-connectivity LSC shell laws.
# ---------------------------------------------------------------------
target_energy_harmonic = 0.5*eps*dim
MAX_METHOD_SECONDS = float(os.environ.get("LEVY_GL_METHOD_TIMEOUT_SECONDS", "1800"))




phase_vector_norm_reference = float(np.mean(np.linalg.norm(phase_vectors, axis=1)))
COHERENCE_MAGNETIZATION_FRACTION = 0.70
COHERENCE_DOMAIN_WALL_MAX = 0.08

def coherence_diagnostics(z):
    M = magnetization(z, n_grid)
    walls = basin_domain_wall_density(z)
    mnorm = np.linalg.norm(M, axis=1)
    coherent = (mnorm >= COHERENCE_MAGNETIZATION_FRACTION * phase_vector_norm_reference) & (walls <= COHERENCE_DOMAIN_WALL_MAX)
    labels = classify_phase(z)
    coherent_mass = np.zeros(n_modes, dtype=float)
    if np.any(coherent):
        coherent_mass = np.bincount(labels[coherent], minlength=n_modes).astype(float)
        coherent_mass = coherent_mass / coherent_mass.sum()
        coherent_tv = float(0.5*np.sum(np.abs(coherent_mass-target_phase_mass)))
    else:
        coherent_tv = np.nan
    return dict(
        coherent_field_fraction=float(np.mean(coherent)),
        noncoherent_field_probability=float(1.0 - np.mean(coherent)),
        coherent_phase_TV=coherent_tv,
        coherence_magnetization_threshold=float(COHERENCE_MAGNETIZATION_FRACTION * phase_vector_norm_reference),
        coherence_domain_wall_threshold=float(COHERENCE_DOMAIN_WALL_MAX),
        **{f"coherent_phase_mass_{k}": float(coherent_mass[k]) for k in range(n_modes)},
    )

def phase_balance_l2(mass): return float(np.linalg.norm(np.asarray(mass)-target_phase_mass))
def radial_order_error(z):
    mq=mean_order_parameter(z); return float(np.mean(np.abs(np.linalg.norm(mq,axis=1)-np.mean(np.linalg.norm(phase_vectors,axis=1)))))
def component_second_moment_error(z):
    mq=mean_order_parameter(z); second=np.mean(mq*mq,axis=0); target=np.mean(phase_vectors*phase_vectors,axis=0); return float(np.linalg.norm(second-target))

def field_observable_row(z):
    M = magnetization(z, n_grid)
    grad_e, local_e = gl_energy_parts(z, n_grid, kappa, local_params=LOCAL_PARAMS)
    corr = vector_correlation(z, n_grid)
    sf = structure_factor(z, n_grid)
    walls = basin_domain_wall_density(z)
    total_e = grad_e + local_e
    return dict(
        magnetization_norm_mean=float(np.mean(np.linalg.norm(M, axis=1))),
        binder_cumulant=float((1.0 - np.mean(np.sum(M*M,axis=1)**2)/(2.0*np.mean(np.sum(M*M,axis=1))**2 + 1e-30))),
        susceptibility=float(susceptibility(M, n_grid, eps)),
        mean_total_energy_density=float(np.mean(total_e)/n_grid),
        mean_gradient_energy_density=float(np.mean(grad_e)/n_grid),
        mean_local_energy_density=float(np.mean(local_e)/n_grid),
        vector_correlation_0=float(corr[0]) if len(corr) else np.nan,
        vector_correlation_1=float(corr[1]) if len(corr) > 1 else np.nan,
        structure_factor_0=float(sf[0]) if len(sf) else np.nan,
        structure_factor_1=float(sf[1]) if len(sf) > 1 else np.nan,
        domain_wall_density_mean=float(np.mean(walls)),
    )

def simulate(method, seed, z0, deadline=None):
    rng = np.random.default_rng(seed)
    z = np.asarray(z0,dtype=float).copy()
    dt = CFG["dt"]
    rows = []
    samples = []
    score_diags = []
    jump_diags = []
    clip_sum = clip_max = 0.0
    jump_step_counts = np.zeros((n_modes,n_modes), dtype=float)
    jump = jump_laws.get(method)
    for step in range(CFG["n_steps"]+1):
        if deadline is not None and time.time() > deadline:
            raise TimeoutError(f"{method} exceeded {MAX_METHOD_SECONDS:.0f} seconds")
        if step % CFG["record_every"] == 0:
            mass = phase_masses(z)
            mean_q = mean_order_parameter(z)
            ev = V_landau(z)
            ent = mode_entropy_metrics(mass)
            row = dict(method=method, seed=seed, time=step*dt,
                phase_TV=float(0.5*np.sum(np.abs(mass-target_phase_mass))),
                phase_balance_l2=phase_balance_l2(mass),
                coverage=int(np.sum(mass>0.02)),
                entropy=ent["entropy"],
                effective_phase_count=ent["effective_mode_count"],
                mean_q1=float(np.mean(mean_q[:,0])), mean_q2=float(np.mean(mean_q[:,1])), mean_order_norm=float(np.linalg.norm(np.mean(mean_q,axis=0))),
                radial_order_error=radial_order_error(z), second_moment_error=component_second_moment_error(z),
                mean_energy=float(np.mean(ev)),
                energy_harmonic_reference_error=float(abs(np.mean(ev)-target_energy_harmonic)),
                energy_std=float(np.std(ev)))
            row.update({f"mass_{k}": float(mass[k]) for k in range(n_modes)})
            nearest_mass = nearest_minimum_masses(z)
            row.update({f"nearest_minimum_diagnostic_mass_{k}": float(nearest_mass[k]) for k in range(n_modes)})
            row["nearest_minimum_diagnostic_TV"] = float(0.5*np.sum(np.abs(nearest_mass-target_phase_mass)))
            row.update(field_observable_row(z))
            row.update(coherence_diagnostics(z))
            rows.append(row)
            samples.append(z.copy())
        if step == CFG["n_steps"]:
            break
        if method == "Langevin":
            drift = -grad_V_landau(z)
            z = z + dt*drift + np.sqrt(2*eps*dt)*rng.standard_normal(z.shape)
        elif method.startswith("LSC-"):
            S, diag = score_for_jump_gl(z, jump)
            score_diags.append(diag)
            drift = -grad_V_landau(z)+S
            z = tamed_euler_step(z, drift, rng, dt)
            before = z.copy()
            before_labels = classify_phase(before)
            z, counts = apply_jump_for_law_gl(z, jump, rng, dt)
            after_labels = classify_phase(z)
            jumped = counts > 0
            if np.any(jumped):
                jump_step_counts += compute_jump_step_transition_matrix(before_labels[jumped], after_labels[jumped], n_modes)
            jd = jump_count_summary(counts)
            jd.update(energy_before_after(V_landau, before, z))
            jump_diags.append(jd)
        else:
            raise ValueError(method)
        z, frac = safety_clip(z)
        clip_sum += frac
        clip_max = max(clip_max, frac)
    df = pd.DataFrame(rows)
    df["clip_fraction_mean"] = clip_sum/max(1,CFG["n_steps"])
    df["clip_fraction_max"] = clip_max
    df["safety_clip_fraction"] = df["clip_fraction_mean"]
    if jump is not None:
        jump_diag = merge_score_diagnostics(jump_diags)
        score_diag = merge_score_diagnostics(score_diags)
        for key, val in jump_diag.items():
            df[key] = val
        for key, val in score_diag.items():
            df[key] = val
            df[f"score_{key}"] = val
        df["jump_count"] = float(jump_diag.get("total_jump_count", 0.0))
        df["undirected_edges"] = jump.edge_count
        df["directed_atoms"] = len(jump.atoms)
        df["graph_family"] = method.split("-")[1]
        df["jump_type"] = getattr(jump, "jump_type", "shell")
        df["h_shell"] = float(getattr(jump, "h_shell", 0.0))
        cost = cost_proxy_fields(CFG["n_steps"], CFG["n_particles"], len(jump.atoms), CFG["theta_n"], CFG["rho_n"], uses_score=True)
        for key, val in cost.items():
            df[key] = val
    else:
        df["jump_count"] = 0.0
        df["mean_jump_count"] = 0.0
        df["undirected_edges"] = 0
        df["directed_atoms"] = 0
        df["graph_family"] = "none"
        df["jump_type"] = "none"
        for key, val in cost_proxy_fields(CFG["n_steps"], CFG["n_particles"], 0, 0, 0, uses_score=False).items():
            df[key] = val
        df["h_shell"] = 0.0
    return df, samples, jump_step_counts

def runtime_text(seconds):
    minutes, sec = divmod(float(seconds), 60.0)
    hours, minutes = divmod(int(minutes), 60)
    return f"{hours:d}:{minutes:02d}:{sec:04.1f}"


def progress_path_value(path):
    path = Path(path)
    return str(path.relative_to(PROJECT_ROOT)) if path.exists() else ""

def write_method_log(method, status, start_time, end_time, elapsed, paths, error=None):
    lines = [
        f"method: {method}",
        f"profile: {PROFILE}",
        f"start_time: {start_time}",
        f"end_time: {end_time}",
        f"runtime_seconds: {elapsed:.6f}",
        f"status: {status}",
        f"csv: {paths['csv']}",
        f"figure: {paths['figure']}",
    ]
    if error is not None:
        lines.append(f"error: {repr(error)}")
    paths["log"].parent.mkdir(parents=True, exist_ok=True)
    paths["log"].write_text("\n".join(lines) + "\n", encoding="utf-8")
    return paths["log"]

def save_method_checkpoint_figure(method, method_metrics, path):
    g_tv = method_metrics.groupby("time")["phase_TV"].agg(["mean", "sem"]).reset_index().fillna(0.0)
    fig, ax = plt.subplots(1, 2, figsize=(10.8, 4.0), constrained_layout=True)
    y = g_tv["mean"].to_numpy()
    se = g_tv["sem"].to_numpy()
    ax[0].semilogy(g_tv.time, np.maximum(y, 1e-12), color=method_color(method), lw=1.8)
    ax[0].fill_between(g_tv.time, np.maximum(y - 2*se, 1e-12), y + 2*se, color=method_color(method), alpha=0.12, lw=0)
    ax[0].set_title(f"{method}: phase TV")
    ax[0].set_xlabel("time")
    ax[0].set_ylabel("TV")
    clean_axes(ax[0])
    for k in range(n_modes):
        gk = method_metrics.groupby("time")[f"mass_{k}"].mean().reset_index()
        ax[1].plot(gk.time, gk[f"mass_{k}"], lw=1.1, label=phase_names[k])
    for kk, mm in enumerate(target_phase_mass):
        ax[1].axhline(mm, color="black", ls="--", lw=0.8, alpha=0.45)
    ax[1].set_ylim(0.0, 1.02)
    ax[1].set_title(f"{method}: phase masses")
    ax[1].set_xlabel("time")
    ax[1].legend(frameon=False, fontsize=6, ncol=2)
    clean_axes(ax[1])
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("saved", path)
    return path

methods = list(PROFILE_METHODS[PROFILE])
selected_gl_method = os.environ.get("LEVY_GL_METHOD", "").strip()
if selected_gl_method:
    if selected_gl_method not in methods:
        raise ValueError(f"Unknown LEVY_GL_METHOD={selected_gl_method!r}; expected one of {methods}")
    methods = [selected_gl_method]
print("active GL methods =", methods)

rng0 = np.random.default_rng(GLOBAL_SEED+9001)
initial_states = []
for s in range(CFG["n_seeds"]):
    initial_states.append(phase_states[0]+0.06*rng0.standard_normal((CFG["n_particles"],dim)))

all_metric_frames = []
sample_store = {}
jump_step_store = {}
method_runtimes = {}
progress_rows = []
progress_file = gl_progress_path(PROJECT_ROOT)
write_gl_progress(progress_file, progress_rows)

overall_t0 = time.time()
for mi, method in enumerate(methods):
    paths = gl_method_output_paths(PROJECT_ROOT, method, PROFILE)
    for p in paths.values():
        p.parent.mkdir(parents=True, exist_ok=True)
    start_time = pd.Timestamp.now().isoformat(timespec="seconds")
    t0_method = time.time()
    deadline = t0_method + MAX_METHOD_SECONDS
    method_results = []
    method_frames = []
    status = "success"
    error = None
    try:
        print(f"starting {method} at {start_time}")
        for s, z0 in enumerate(initial_states):
            seed = GLOBAL_SEED + 1000*s + 17*mi
            df, samples, jump_counts = simulate(method, seed, z0, deadline=deadline)
            method_results.append((seed, samples, jump_counts))
            method_frames.append(df)
        elapsed = time.time() - t0_method
        method_metrics = pd.concat(method_frames, ignore_index=True)
        method_metrics.to_csv(paths["csv"], index=False)
        save_method_checkpoint_figure(method, method_metrics, paths["figure"])
        for seed, samples, jump_counts in method_results:
            sample_store[(method, seed)] = samples
            jump_step_store[(method, seed)] = jump_counts
        all_metric_frames.append(method_metrics)
    except Exception as exc:
        elapsed = time.time() - t0_method
        status = "timed_out" if isinstance(exc, TimeoutError) else "failed"
        error = exc
        if method_frames:
            partial_metrics = pd.concat(method_frames, ignore_index=True)
            partial_metrics.to_csv(paths["csv"], index=False)
        end_time = pd.Timestamp.now().isoformat(timespec="seconds")
        write_method_log(method, status, start_time, end_time, elapsed, paths, error=error)
        progress_rows.append({
            "Method": method,
            "Start time": start_time,
            "End time": end_time,
            "Runtime": runtime_text(elapsed),
            "Status": status,
            "Output CSV": progress_path_value(paths["csv"]),
            "Output figure": progress_path_value(paths["figure"]),
            "Log": progress_path_value(paths["log"]),
        })
        write_gl_progress(progress_file, progress_rows)
        print(f"{method} {status} after {elapsed:.2f} seconds")
        raise
    end_time = pd.Timestamp.now().isoformat(timespec="seconds")
    method_runtimes[method] = elapsed
    write_method_log(method, status, start_time, end_time, elapsed, paths)
    progress_rows.append({
        "Method": method,
        "Start time": start_time,
        "End time": end_time,
        "Runtime": runtime_text(elapsed),
        "Status": status,
        "Output CSV": progress_path_value(paths["csv"]),
        "Output figure": progress_path_value(paths["figure"]),
        "Log": progress_path_value(paths["log"]),
    })
    write_gl_progress(progress_file, progress_rows)
    print(f"{method} completed in {elapsed:.2f} seconds")

print("simulation elapsed seconds:", time.time()-overall_t0)
metrics = pd.concat(all_metric_frames, ignore_index=True)
metrics.to_csv(TABDIR/"vector_gl_metrics_timeseries.csv", index=False)
display(metrics.groupby("method")[["phase_TV","coherent_phase_TV","coherent_field_fraction","noncoherent_field_probability","coverage","effective_phase_count","mean_order_norm","magnetization_norm_mean","energy_harmonic_reference_error","domain_wall_density_mean"]].tail(1))

def first_crossing_time(times, vals, tol):
    times=np.asarray(times); vals=np.asarray(vals); ok=np.isfinite(vals)&(vals<=tol)
    return (float(times[np.where(ok)[0][0]]), True) if np.any(ok) else (np.nan, False)

mix_rows=[]
for method in methods:
    sub=metrics[metrics.method==method].groupby("time")["phase_TV"].mean().reset_index()
    tmix,reached=first_crossing_time(sub.time, sub.phase_TV, CFG["mixing_tol"])
    terminal=float(sub.phase_TV.iloc[-1])
    cost=float(metrics[metrics.method==method]["total_cost_proxy"].iloc[0]) if "total_cost_proxy" in metrics else np.nan
    mix_rows.append(dict(method=method, mixing_tol=CFG["mixing_tol"], mixing_time=tmix, reached=reached,
                         terminal_phase_TV=terminal, total_cost_proxy=cost,
                         runtime_seconds=float(method_runtimes.get(method, np.nan)),
                         mixing_time_x_total_cost_proxy=float(tmix*cost) if np.isfinite(tmix) and np.isfinite(cost) else np.nan))
mixing_df=pd.DataFrame(mix_rows).merge(graph_df[["method","graph_family","undirected_edges","directed_atoms","graph_gap","jump_type","h_shell"]], on="method", how="left")
mixing_df.to_csv(TABDIR/"vector_gl_mixing_times.csv", index=False)
display(mixing_df)


In [ ]:
# ---------------------------------------------------------------------
# Phase communication, mass evolution, and graph-cost figures.
# ---------------------------------------------------------------------
def mean_sem(metric):
    return metrics.groupby(["method", "time"])[metric].agg(["mean", "sem"]).reset_index().fillna(0.0)

fig, ax = plt.subplots(1, 3, figsize=(15.8, 4.6), constrained_layout=True)
for metric, title, ylabel, axi, logy in [
    ("phase_TV", "phase-reference TV", "TV", ax[0], True),
    ("effective_phase_count", "effective phase count", "count", ax[1], False),
    ("domain_wall_density_mean", "domain-wall density", "fraction of neighboring sites", ax[2], False),
]:
    g = mean_sem(metric)
    for method in methods:
        sub = g[g.method == method]
        y = sub["mean"].to_numpy()
        se = sub["sem"].to_numpy()
        if logy:
            yy = np.maximum(y, 1e-12)
            axi.semilogy(sub.time, yy, color=method_color(method), lw=1.8, label=method)
            axi.fill_between(sub.time, np.maximum(yy-2*se, 1e-12), yy+2*se,
                             color=method_color(method), alpha=0.10, lw=0)
        else:
            axi.plot(sub.time, y, color=method_color(method), lw=1.8, label=method)
            axi.fill_between(sub.time, y-2*se, y+2*se,
                             color=method_color(method), alpha=0.10, lw=0)
    axi.set_title(title)
    axi.set_xlabel("time")
    axi.set_ylabel(ylabel)
    clean_axes(axi)
for j, axi in enumerate(ax):
    panel_label(axi, "abc"[j])
ax[0].legend(frameon=False, fontsize=7, ncol=2)
savefig(fig, "fig02_coupled_phi4_phase_communication")
plt.show()

# Four phase-mass trajectories, a grouped final mass comparison, and phase-reference TV.
fig, ax = plt.subplots(2, 3, figsize=(15.8, 8.8), constrained_layout=True)
axes = ax.ravel()
for k in range(n_modes):
    axi = axes[k]
    g = metrics.groupby(["method", "time"])[f"mass_{k}"].agg(["mean", "sem"]).reset_index().fillna(0.0)
    for method in methods:
        sub = g[g.method == method]
        axi.plot(sub.time, sub["mean"], color=method_color(method), lw=1.6, label=method)
    axi.axhline(target_phase_mass[k], color="black", ls="--", lw=1.0)
    axi.set_title(f"global phase mass: {phase_names[k]}")
    axi.set_ylim(0, 1.02)
    axi.set_xlabel("time")
    clean_axes(axi)

final = metrics.sort_values("time").groupby(["method", "seed"]).tail(1)
x = np.arange(n_modes)
width = min(0.12, 0.72/max(1, len(methods)))
for j, method in enumerate(methods):
    subf = final[final.method == method]
    means = np.array([subf[f"mass_{k}"].mean() for k in range(n_modes)])
    ses = np.array([subf[f"mass_{k}"].sem() for k in range(n_modes)])
    axes[4].bar(x+(j-(len(methods)-1)/2)*width, means, width=width,
                yerr=np.nan_to_num(ses), capsize=2, color=method_color(method),
                label=method, alpha=0.80)
axes[4].plot(x, target_phase_mass, "k--", lw=2.0, label="Laplace phase reference")
axes[4].set_xticks(x)
axes[4].set_xticklabels(phase_names)
axes[4].set_ylabel("probability mass")
axes[4].set_title("final global phase masses")
clean_axes(axes[4])

g = mean_sem("phase_TV")
for method in methods:
    sub = g[g.method == method]
    axes[5].semilogy(sub.time, np.maximum(sub["mean"], 1e-12),
                     color=method_color(method), lw=1.8, label=method)
axes[5].set_title("phase-reference TV")
axes[5].set_xlabel("time")
axes[5].set_ylabel("TV")
clean_axes(axes[5])

for j, axi in enumerate(axes):
    panel_label(axi, "abcdef"[j])
axes[0].legend(frameon=False, fontsize=7, ncol=2)
axes[4].legend(frameon=False, fontsize=6, ncol=2)
savefig(fig, "fig03_coupled_phi4_phase_mass_and_tv")
plt.show()

# Graph gap / edge cost / mixing time.
fig, ax = plt.subplots(1, 3, figsize=(15.0, 4.6), constrained_layout=True)
sub = mixing_df[mixing_df.method != "Langevin"].sort_values("undirected_edges")
ax[0].plot(sub.undirected_edges, sub.graph_gap, marker="o", lw=2.0)
ax[0].set_title("coarse graph spectral gap")
ax[0].set_xlabel("undirected edge count")
ax[0].set_ylabel("graph gap")
ax[1].plot(sub.undirected_edges, sub.mixing_time, marker="o", lw=2.0)
ax[1].set_title("empirical phase-reference TV mixing time")
ax[1].set_xlabel("undirected edge count")
ax[1].set_ylabel("time")
ax[2].plot(sub.undirected_edges, sub.mixing_time*sub.undirected_edges, marker="o", lw=2.0)
ax[2].set_title("cost-adjusted mixing time")
ax[2].set_xlabel("undirected edge count")
ax[2].set_ylabel("mixing time x edges")
for j, axi in enumerate(ax):
    clean_axes(axi)
    panel_label(axi, "abc"[j])
savefig(fig, "fig04_vector_landau_graph_gap_and_cost")
plt.show()


In [ ]:
# ---------------------------------------------------------------------
# Final distributions, transition matrices, diagnostics, and profile heatmaps.
# ---------------------------------------------------------------------
def get_first_key(method): return next(k for k in sample_store if k[0]==method)
final=metrics.sort_values('time').groupby(['method','seed']).tail(1)
final.to_csv(TABDIR/'vector_gl_final_metrics_by_seed.csv', index=False)

fig,ax=plt.subplots(2,3,figsize=(16,8.6),constrained_layout=True)
for method in methods:
    key=get_first_key(method); zfinal=sample_store[key][-1]; mq=mean_order_parameter(zfinal); ev=V_landau(zfinal)
    ax[0,0].hist(ev,bins=34,histtype='step',lw=1.5,color=method_color(method),label=method,density=True)
    ax[0,1].hist(mq[:,0],bins=34,histtype='step',lw=1.5,color=method_color(method),density=True)
    ax[0,2].hist(mq[:,1],bins=34,histtype='step',lw=1.5,color=method_color(method),density=True)
    ax[1,0].hist(np.linalg.norm(mq,axis=1),bins=34,histtype='step',lw=1.5,color=method_color(method),density=True)
    ax[1,1].scatter(mq[:,0],mq[:,1],s=7,alpha=0.22,color=method_color(method),label=method)
ax[0,0].axvline(target_energy_harmonic,color='black',ls='--',lw=1.0); ax[0,0].set_title('final energy distribution')
ax[0,1].set_title('final mean u1 distribution'); ax[0,2].set_title('final mean u2 distribution'); ax[1,0].set_title('final |mean u| distribution')
ax[1,1].scatter(phase_vectors[:,0],phase_vectors[:,1],s=120,facecolor='white',edgecolor='black',zorder=5)
for k,p in enumerate(phase_vectors): ax[1,1].text(p[0],p[1]+0.08,phase_names[k],ha='center',fontsize=8)
ax[1,1].set_xlim(-1.35,1.35); ax[1,1].set_ylim(-1.35,1.35); ax[1,1].set_aspect('equal'); ax[1,1].set_title('final mean-order scatter')
x=np.arange(n_modes); width=min(0.12, 0.72/max(1,len(methods)))
for j,method in enumerate(methods):
    subf=final[final.method==method]; means=np.array([subf[f'mass_{k}'].mean() for k in range(n_modes)])
    ax[1,2].bar(x+(j-(len(methods)-1)/2)*width,means,width=width,color=method_color(method),label=method,alpha=0.75)
ax[1,2].plot(x, target_phase_mass, 'k--', lw=2.0, label='Laplace phase reference'); ax[1,2].set_xticks(x); ax[1,2].set_xticklabels(phase_names,rotation=30,ha='right'); ax[1,2].set_title('final phase occupancy')
for j,axi in enumerate(ax.ravel()): clean_axes(axi); panel_label(axi,'abcdef'[j])
ax[0,0].legend(frameon=False,fontsize=6,ncol=2)
savefig(fig,'fig05_vector_landau_final_distributions'); plt.show()

# transition matrices between recorded snapshots
trans_rows=[]
for method in methods:
    acc=np.zeros((n_modes,n_modes),dtype=float)
    for key,slist in sample_store.items():
        if key[0]!=method: continue
        label_series=[classify_phase(arr) for arr in slist]
        acc += compute_recorded_phase_transition_matrix(label_series, n_modes)
    trans_rows.extend(transition_rows_from_counts(acc, phase_names, method, "recorded_phase"))
transition_df=pd.DataFrame(trans_rows)
transition_df.to_csv(TABDIR/'vector_gl_recorded_phase_transition_matrix.csv',index=False)
fig,ax=plt.subplots(1,len(methods),figsize=(3.0*len(methods),3.4),constrained_layout=True)
if len(methods) == 1:
    ax=[ax]
for mi,method in enumerate(methods):
    mat=transition_df[transition_df.method==method].pivot(index='from_state',columns='to_state',values='probability').loc[phase_names,phase_names].to_numpy()
    im=ax[mi].imshow(mat,vmin=0,vmax=1,cmap='magma'); ax[mi].set_title(method, fontsize=8); ax[mi].set_xticks(range(n_modes)); ax[mi].set_xticklabels(phase_names,rotation=35,fontsize=7); ax[mi].set_yticks(range(n_modes)); ax[mi].set_yticklabels(phase_names,fontsize=7)
    for i in range(n_modes):
        for j in range(n_modes): ax[mi].text(j,i,f'{mat[i,j]:.2f}',ha='center',va='center',color='white' if mat[i,j]>0.45 else 'black',fontsize=6)
fig.colorbar(im,ax=np.ravel(ax).tolist(),fraction=0.025,pad=0.02)
savefig(fig,'fig06_vector_landau_recorded_phase_transition_matrices'); plt.show()

jump_rows=[]
for (method, seed), C in jump_step_store.items():
    jump_rows.extend(dict(row, seed=seed) for row in transition_rows_from_counts(C, phase_names, method, "jump_step_phase"))
jump_transition_df=pd.DataFrame(jump_rows)
jump_transition_df.to_csv(TABDIR/'vector_gl_jump_step_transition_matrix.csv',index=False)

# sampler diagnostics
diag=[]
for method in methods:
    subm=metrics[metrics.method==method]
    row=dict(method=method,
             graph_family=str(subm.graph_family.iloc[0]) if 'graph_family' in subm else 'none',
             jump_type=str(subm.jump_type.iloc[0]) if 'jump_type' in subm else 'none',
             clip_fraction_mean=float(subm.clip_fraction_mean.mean()),
             clip_fraction_max=float(subm.clip_fraction_max.max()),
             mean_jump_count=float(subm.mean_jump_count.dropna().mean()) if 'mean_jump_count' in subm else 0.0,
             total_jump_count=float(subm.jump_count.dropna().mean()) if 'jump_count' in subm else 0.0,
             undirected_edges=float(subm.undirected_edges.mean()) if 'undirected_edges' in subm else 0.0,
             n_atoms=float(subm.n_atoms.mean()) if 'n_atoms' in subm else 0.0,
             n_theta=float(subm.n_theta.mean()) if 'n_theta' in subm else 0.0,
             n_rho=float(subm.n_rho.mean()) if 'n_rho' in subm else 0.0,
             rho_n=float(subm.rho_n.mean()) if 'rho_n' in subm else 0.0,
             h_shell=float(subm.h_shell.mean()) if 'h_shell' in subm else 0.0,
             jump_cost_proxy=float(subm.jump_cost_proxy.mean()) if 'jump_cost_proxy' in subm else 0.0,
             score_cost_proxy=float(subm.score_cost_proxy.mean()) if 'score_cost_proxy' in subm else 0.0,
             total_cost_proxy=float(subm.total_cost_proxy.mean()) if 'total_cost_proxy' in subm else 0.0,
             cost_proxy=float(subm.cost_proxy.mean()) if 'cost_proxy' in subm else 0.0)
    for col in ['score_clip_fraction','score_changing_logratio_clip_fraction','overflow_guard_logratio_clip_fraction','effective_log_clip','logratio_clip_fraction','safety_clip_fraction','max_score_norm','max_log_ratio','mean_energy_before_jump','mean_energy_after_jump']:
        row[col]=float(subm[col].dropna().mean()) if col in subm and subm[col].notna().any() else np.nan
    diag.append(row)
diag_df=pd.DataFrame(diag); diag_df.to_csv(TABDIR/'vector_gl_sampler_diagnostics.csv',index=False); display(diag_df)
fig,ax=plt.subplots(1,4,figsize=(17.0,4.2),constrained_layout=True)
ax[0].bar(diag_df.method,diag_df.clip_fraction_max,color=[method_color(m) for m in diag_df.method]); ax[0].set_title('max safety clipping')
ax[1].bar(diag_df.method,diag_df.mean_jump_count,color=[method_color(m) for m in diag_df.method]); ax[1].set_title('mean jumps per particle-step')
ax[2].bar(diag_df.method,diag_df.score_clip_fraction.fillna(0.0),color=[method_color(m) for m in diag_df.method]); ax[2].set_title('score clipping fraction')
ax[3].bar(diag_df.method,diag_df.score_changing_logratio_clip_fraction.fillna(0.0),color=[method_color(m) for m in diag_df.method]); ax[3].set_title('score-changing log-ratio clipping')
for j,axi in enumerate(ax): axi.tick_params(axis='x',rotation=65,labelsize=6); clean_axes(axi); panel_label(axi,'abcd'[j])
savefig(fig,'fig07_vector_landau_sampler_diagnostics'); plt.show()

# Physical field diagnostics: direct observables used for interpretation, not
# critical-scaling claims.
fig, ax = plt.subplots(2, 3, figsize=(15.8, 8.4), constrained_layout=True)
physical_specs = [
    ("mean_total_energy_density", "total energy density"),
    ("mean_gradient_energy_density", "gradient energy density"),
    ("mean_local_energy_density", "local energy density"),
    ("domain_wall_density_mean", "domain-wall density"),
    ("binder_cumulant", "Binder cumulant"),
    ("susceptibility", "susceptibility"),
]
for j, (metric, title) in enumerate(physical_specs):
    g = metrics.groupby(["method", "time"])[metric].agg(["mean", "sem"]).reset_index().fillna(0.0)
    for method in methods:
        sub = g[g.method == method]
        ax.ravel()[j].plot(sub.time, sub["mean"], color=method_color(method), lw=1.7, label=method)
    ax.ravel()[j].set_title(title)
    ax.ravel()[j].set_xlabel("time")
    clean_axes(ax.ravel()[j])
    panel_label(ax.ravel()[j], "abcdef"[j])
ax.ravel()[0].legend(frameon=False, fontsize=7, ncol=2)
savefig(fig, "fig09_coupled_phi4_physical_diagnostics")
plt.show()

# final heatmaps
fig,ax=plt.subplots(len(methods),2,figsize=(12.8,2.0*len(methods)),constrained_layout=True)
if len(methods) == 1:
    ax = np.asarray([ax])
for r,method in enumerate(methods):
    key=get_first_key(method); zfinal=sample_store[key][-1][:min(90,CFG['n_particles'])]; labs=classify_phase(zfinal); order=np.argsort(labs); q=unpack(zfinal[order])
    im0=ax[r,0].imshow(q[:,:,0],aspect='auto',cmap='coolwarm',vmin=-1.25,vmax=1.25,extent=[0,1,len(q),0])
    im1=ax[r,1].imshow(q[:,:,1],aspect='auto',cmap='coolwarm',vmin=-1.25,vmax=1.25,extent=[0,1,len(q),0])
    ax[r,0].set_title(f'{method}: final u1 samples', fontsize=8); ax[r,1].set_title(f'{method}: final u2 samples', fontsize=8); ax[r,0].set_ylabel('sample index')
    ax[r,0].set_xlabel('x'); ax[r,1].set_xlabel('x')
fig.colorbar(im0,ax=ax[:,0],fraction=0.025,pad=0.02); fig.colorbar(im1,ax=ax[:,1],fraction=0.025,pad=0.02)
savefig(fig,'fig08_vector_landau_final_profile_heatmaps'); plt.show()

## Interpretation

The coupled phi4 example separates local geometry from global field phases.
The local gradient-flow basin map and local basin masses verify that all four
sign-state wells remain present.  The global field phase is obtained by
looking up the mean order parameter in the same basin map, and its phase-reference TV is
measured against a coherent-field Laplace phase-mass reference.

The main comparison concerns phase communication across graph families.
Energy density, domain-wall density, Binder cumulant, and susceptibility are
reported as finite-parameter physical diagnostics.  Correlation and structure
factor columns remain available in the tables, but no critical-scaling claim
is made from this single lattice size and temperature.


## Output registry

In [ ]:
# Phase17C output registry and run summary.
from datetime import datetime
import json

table_files = sorted(str(p.relative_to(PROJECT_ROOT)) for p in (RELEASE_TABLE_DIR / "04_coupled_phi4_gl").glob("*.csv"))
figure_files = sorted(str(p.relative_to(PROJECT_ROOT)) for p in DIAGNOSTIC_FIG_DIR.glob(f"{FIG_PREFIX}*.pdf"))
registry = {
    "experiment": "coupled double-well / two-component phi4 GL",
    "table_directory": str((RELEASE_TABLE_DIR / "04_coupled_phi4_gl").relative_to(PROJECT_ROOT)),
    "figure_directory": str(DIAGNOSTIC_FIG_DIR.relative_to(PROJECT_ROOT)),
    "tables": table_files,
    "figures": figure_files,
    "created_utc_like": datetime.utcnow().isoformat(timespec="seconds") + "Z",
}
registry_path = RELEASE_LOG_DIR / "04_coupled_phi4_gl_output_registry.json"
registry_path.write_text(json.dumps(registry, indent=2) + "\n", encoding="utf-8")
pd.DataFrame([{"experiment": "coupled double-well / two-component phi4 GL", "n_tables": len(table_files), "n_figures": len(figure_files)}]).to_csv(
    RELEASE_TABLE_DIR / "04_coupled_phi4_gl_summary.csv", index=False
)
print("Phase17C registry written:", registry_path)
print("tables:", len(table_files), "figures:", len(figure_files))


In [ ]:
SCRIPTS_DIR = RELEASE_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from generate_canonical_release_figures import generate_gl_release
generate_gl_release()